# Stage 1 — Movie Review Sentiment Classifier

**Labels:** `0 = negative`, `1 = positive`.

The training set contains 240 reviews (180 positive, 60 negative). The public test set contains 400 reviews (200 positive, 200 negative).

## Model structure

I use a **CountVectorizer + Logistic Regression** pipeline.

**Review → word-count features → Logistic Regression → 0/1**

CountVectorizer converts each review into word-occurrence features. I use word unigrams because the training set is very small, which keeps the feature space simpler and reduces overfitting. Logistic Regression uses L2 regularization and balanced class weights.

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

In [ ]:
train_df = pd.read_csv("train.csv")
public_test_df = pd.read_csv("public_test.csv")

print("Training shape:", train_df.shape)
print("Public test shape:", public_test_df.shape)
print("\nTraining labels:")
print(train_df["label"].value_counts())
print("\nPublic test labels:")
print(public_test_df["label"].value_counts())

## Handling the small and imbalanced training set

The training data is small and has a 3:1 positive-to-negative imbalance.

- I use a **simple linear classifier** instead of a neural network to reduce overfitting.
- **L2 regularization** controls the magnitude of model coefficients.
- `class_weight="balanced"` gives additional importance to the minority negative class.
- A **stratified validation split** preserves the class proportions.
- The final model is retrained using all 240 labeled reviews after the model configuration is selected.

In [ ]:
X = train_df["text"].fillna("").astype(str)
y = train_df["label"].astype(int)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Training distribution:\n", y_train.value_counts())
print("Validation distribution:\n", y_val.value_counts())

## Key training techniques

- **Feature representation:** word-count unigrams (`ngram_range=(1,1)`).
- **Regularization:** L2, with `C=1.0`.
- **Class imbalance:** `class_weight="balanced"`.
- **Solver/optimizer:** `liblinear`, suitable for a small binary classification problem.
- **Learning rate:** no manual learning rate is used because `liblinear` is a conventional Logistic Regression solver rather than a mini-batch gradient-descent implementation.
- **Batch size:** not applicable to this solver.
- **Maximum iterations:** 1000.
- **Random state:** 42 for reproducibility.

In [ ]:
def make_model():
    return Pipeline([
        ("vectorizer", CountVectorizer(
            lowercase=True,
            strip_accents="unicode",
            ngram_range=(1, 1),
            min_df=1,
            max_features=20000
        )),
        ("classifier", LogisticRegression(
            C=1.0,
            class_weight="balanced",
            solver="liblinear",
            max_iter=1000,
            random_state=42
        ))
    ])

model = make_model()
model.fit(X_train, y_train)

val_predictions = model.predict(X_val)
val_accuracy = accuracy_score(y_val, val_predictions)
val_cm = confusion_matrix(y_val, val_predictions)

print("Validation accuracy:", val_accuracy)
print("Validation confusion matrix:")
print(val_cm)

In [ ]:
ConfusionMatrixDisplay(
    confusion_matrix=val_cm,
    display_labels=["Negative", "Positive"]
).plot()
plt.title("Validation Confusion Matrix")
plt.show()

## Final model

After selecting the model structure, the final model is trained on **all 240 training reviews**. This gives the classifier access to every labeled training example before public-test evaluation.

In [ ]:
final_model = make_model()
final_model.fit(X, y)

X_public_test = public_test_df["text"].fillna("").astype(str)
y_public_test = public_test_df["label"].astype(int)

public_predictions = final_model.predict(X_public_test)
public_accuracy = accuracy_score(y_public_test, public_predictions)
public_cm = confusion_matrix(y_public_test, public_predictions)

print("Public test accuracy:", public_accuracy)
print("Public test confusion matrix:")
print(public_cm)

In [ ]:
ConfusionMatrixDisplay(
    confusion_matrix=public_cm,
    display_labels=["Negative", "Positive"]
).plot()
plt.title("Public Test Confusion Matrix")
plt.show()

print(classification_report(
    y_public_test,
    public_predictions,
    target_names=["Negative", "Positive"],
    digits=4
))

## Evaluation tokens not shown during training

The vectorizer is fitted **only on the training reviews**. It never uses the public-test reviews to build its vocabulary.

If a word appears in a public-test review but did not occur in training, CountVectorizer simply does not create a feature for that word. Other known words in the review can still be used for prediction.

This prevents test-set vocabulary information from leaking into training.

## Public test evaluation result

The final model achieved **0.6750 (67.50%) accuracy** on the 400 public-test reviews.

### Confusion matrix

| | Predicted Negative | Predicted Positive |
|---|---:|---:|
| **Actual Negative** | 100 | 100 |
| **Actual Positive** | 30 | 170 |

## Save and reload the model checkpoint

The complete Pipeline is saved in `model_checkpoint/sentiment_model.joblib`. It contains the fitted vocabulary and the fitted Logistic Regression classifier.

In [ ]:
os.makedirs("model_checkpoint", exist_ok=True)

joblib.dump(final_model, "model_checkpoint/sentiment_model.joblib")

loaded_model = joblib.load("model_checkpoint/sentiment_model.joblib")
reloaded_predictions = loaded_model.predict(X_public_test)

print("Reloaded predictions identical:",
      np.array_equal(public_predictions, reloaded_predictions))

## Create `public_test_predictions.csv`

The required file contains exactly:

`id,predicted_label`

with `predicted_label` equal to either `0` or `1`.

In [ ]:
prediction_df = pd.DataFrame({
    "id": public_test_df["id"],
    "predicted_label": public_predictions.astype(int)
})

prediction_df.to_csv("public_test_predictions.csv", index=False)

print(prediction_df.head())
print("Columns:", prediction_df.columns.tolist())
print("Rows:", len(prediction_df))